In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

RAW = Path("../data/raw")
PROC = Path("../data/processed")

PRICES_XLSX = RAW / "wfp-2016-2026.xlsx"
MARKETS_CSV = RAW / "wfp_markets_phl.csv"

print("Prices file exists :", PRICES_XLSX.exists())
print("Markets file exists:", MARKETS_CSV.exists())

Prices file exists : True
Markets file exists: True


In [2]:
prices = pd.read_excel(PRICES_XLSX, sheet_name=0)
markets = pd.read_csv(MARKETS_CSV)

# Convert once to CSV so later notebooks load in seconds
prices.to_csv(PROC / "wfp_food_prices_phl.csv", index=False)

print("Prices :", prices.shape)
print("Markets:", markets.shape)

Prices : (210339, 16)
Markets: (132, 7)


In [3]:
print("=== PRICES: head ===")
display(prices.head())
print("\n=== PRICES: dtypes ===")
print(prices.dtypes)

print("\n=== MARKETS: head ===")
display(markets.head())
print("\n=== MARKETS: dtypes ===")
print(markets.dtypes)

=== PRICES: head ===


,date,admin1,admin2,market,market_id,latitude,longitude,category,commodity,commodity_id,unit,priceflag,pricetype,currency,price,usdprice
0,2016-01-15,Autonomous region in Muslim Mindanao,Maguindanao,Shariff Aguak,183,6.86,124.44,cereals and tubers,Potatoes (Irish),148,KG,actual,Retail,PHP,73.42,1.54
1,2016-01-15,Autonomous region in Muslim Mindanao,Maguindanao,Shariff Aguak,183,6.86,124.44,cereals and tubers,"Rice (regular, milled)",80,KG,actual,Retail,PHP,36.93,0.77
2,2016-01-15,Autonomous region in Muslim Mindanao,Maguindanao,Shariff Aguak,183,6.86,124.44,cereals and tubers,Sweet potatoes,87,KG,actual,Retail,PHP,33.00,0.69
3,2016-01-15,Autonomous region in Muslim Mindanao,Maguindanao,Shariff Aguak,183,6.86,124.44,"meat, fish and eggs",Eggs,92,Unit,actual,Retail,PHP,8.15,0.17
4,2016-01-15,Autonomous region in Muslim Mindanao,Maguindanao,Shariff Aguak,183,6.86,124.44,"meat, fish and eggs",Fish (fresh),401,KG,actual,Retail,PHP,124.00,2.59



=== PRICES: dtypes ===
date            datetime64[us]
admin1                     str
admin2                     str
market                     str
market_id                int64
latitude               float64
longitude              float64
category                   str
commodity                  str
commodity_id             int64
unit                       str
priceflag                  str
pricetype                  str
currency                   str
price                  float64
usdprice               float64
dtype: object

=== MARKETS: head ===


,market_id,market,countryiso3,admin1,admin2,latitude,longitude
0,167,Metro Manila,PHL,National Capital region,Metropolitan Manila,14.60,120.98
1,168,La Trinidad,PHL,Cordillera Administrative region,Benguet,16.45,120.59
2,169,Lingayen,PHL,Region I,Pangasinan,16.02,120.23
3,170,Tuguegarao City,PHL,Region II,Cagayan,17.61,121.73
4,171,Palayan,PHL,Region III,Nueva Ecija,15.54,121.08



=== MARKETS: dtypes ===
market_id        int64
market             str
countryiso3        str
admin1             str
admin2             str
latitude       float64
longitude      float64
dtype: object


In [4]:
# 1. Expected columns present?
expected_prices = {"date","admin1","admin2","market","market_id","latitude","longitude",
                   "category","commodity","commodity_id","unit","priceflag",
                   "pricetype","currency","price","usdprice"}
expected_markets = {"market_id","market","countryiso3","admin1","admin2","latitude","longitude"}

print("Prices missing cols :", expected_prices - set(prices.columns))
print("Markets missing cols:", expected_markets - set(markets.columns))

# 2. Date parsing and coverage
prices["date"] = pd.to_datetime(prices["date"])
print("\nDate range:", prices["date"].min().date(), "->", prices["date"].max().date())
print("\nRows per year:")
print(prices["date"].dt.year.value_counts().sort_index())

# 3. Missing values
print("\nMissing values (prices):")
print(prices.isna().sum())
print("\nMissing values (markets):")
print(markets.isna().sum())

# 4. Duplicates
print("\nFully duplicated rows:", prices.duplicated().sum())
key = ["date","market_id","commodity_id","unit","pricetype"]
print("Duplicate key rows    :", prices.duplicated(subset=key).sum())
print("market_id unique in markets file:", markets["market_id"].is_unique)

# 5. Categorical profile
for col in ["pricetype","currency","priceflag","category","unit"]:
    print(f"\n{col}:")
    print(prices[col].value_counts())

# 6. Cardinality
print("\nUnique markets    :", prices["market"].nunique())
print("Unique commodities:", prices["commodity"].nunique())
print("Unique admin1     :", prices["admin1"].nunique())

# 7. Referential integrity: prices vs markets
p_ids, m_ids = set(prices["market_id"]), set(markets["market_id"])
print("\nmarket_ids in prices but NOT in markets:", len(p_ids - m_ids))
print("market_ids in markets but NOT in prices:", len(m_ids - p_ids))

Prices missing cols : set()
Markets missing cols: set()

Date range: 2016-01-15 -> 2026-06-15

Rows per year:
date
2016     2631
2017     3339
2018     3583
2019     1636
2020    30700
2021    31136
2022    30155
2023    30216
2024    30240
2025    30965
2026    15738
Name: count, dtype: int64

Missing values (prices):
date            0
admin1          0
admin2          0
market          0
market_id       0
latitude        0
longitude       0
category        0
commodity       0
commodity_id    0
unit            0
priceflag       0
pricetype       0
currency        0
price           0
usdprice        0
dtype: int64

Missing values (markets):
market_id      0
market         0
countryiso3    0
admin1         0
admin2         0
latitude       0
longitude      0
dtype: int64

Fully duplicated rows: 0
Duplicate key rows    : 0
market_id unique in markets file: True

pricetype:
pricetype
Retail       207724
Wholesale      2615
Name: count, dtype: int64

currency:
currency
PHP    210339
Name: 

In [5]:
summary = pd.DataFrame({
    "metric": ["rows","columns","start_date","end_date","n_markets","n_commodities",
               "n_regions","duplicate_keys","missing_cells"],
    "value": [len(prices), prices.shape[1], prices["date"].min().date(), prices["date"].max().date(),
              prices["market"].nunique(), prices["commodity"].nunique(),
              prices["admin1"].nunique(), prices.duplicated(subset=key).sum(),
              int(prices.isna().sum().sum())]
})
summary.to_csv("../outputs/tables/01_validation_summary.csv", index=False)
display(summary)

,metric,value
0,rows,210339
1,columns,16
2,start_date,2016-01-15
3,end_date,2026-06-15
4,n_markets,97
5,n_commodities,66
6,n_regions,17
7,duplicate_keys,0
8,missing_cells,0
